# Mismo mes, otro año

Los profesores sugirieron comparar **el mismo mes de años distintos** (enero de 2025 contra enero de 2026, y así). Aquí se prueba una versión concreta de esa idea:

> Para predecir un mes (digamos marzo de 2026), ¿conviene entrenar con **el mes inmediato anterior** (febrero de 2026) o con **el mismo mes del año anterior** (marzo de 2025)? ¿Y con los dos juntos?

Los tres casos usan **un solo mes de entrenamiento** (los dos juntos, dos), así que la comparación es justa en cantidad de datos. Cada tabla se construye por separado, con sus propios rezagos de 7 días o más, igual que en `02_modelo.ipynb`.

- **Meses de prueba:** enero a septiembre de 2026 (contra 2025) y septiembre a diciembre de 2025 (contra 2024).
- **Modelos:** XGBoost con los mismos hiperparámetros de la versión 1 (sin búsqueda nueva), regresión lineal, promedio histórico y media de 4 semanas. El MLP está apagado por defecto (`USAR_MLP`).
- **Resultados parciales:** el notebook guarda `resultados_mismo_mes.csv` después de cada mes y, si se interrumpe, al volver a correrlo **retoma donde se quedó**.
- **Qué meses corre:** solo los que tienen sus archivos (ver la tabla de la sección 1). Los CSV de 2024 y de enero–agosto de 2025 se bajan aparte; el checklist está en `docs/colab_checklist.md`.

**Cómo correrlo:** Restart & Run All, desde la carpeta `notebooks/`. En Colab, ver el checklist.

In [ ]:
import os
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Colab: la carpeta del proyecto se copia a /content (ver docs/colab_checklist.md) ---
EN_COLAB = "google.colab" in sys.modules
RAIZ_COLAB = "/content/ecobici-demanda"
COPIA_A_DRIVE = "/content/drive/MyDrive/ecobici-demanda/resultados"   # se copia el CSV aquí después de cada mes (si existe)
if EN_COLAB:
    os.chdir(os.path.join(RAIZ_COLAB, "notebooks"))
sys.path.append("../src")

import modelo as M
import mismo_mes as MM

PRUEBAS = [f"2026-{m:02d}" for m in range(1, 10)] + [f"2025-{m:02d}" for m in range(9, 13)]
USAR_LINEAL = True
USAR_MLP = False        # el MLP tarda bastante más; actívalo solo si hace falta
REHACER = False         # True: ignora resultados guardados y corre todo de nuevo
SALIDA = "../data/processed/resultados_mismo_mes.csv"

In [ ]:
dispositivo = MM.configurar_dispositivo()
estaciones = M.cargar_estaciones()
params = MM.cargar_params()
print("XGBoost corre en:", dispositivo)
print("Hiperparámetros (versión 1):", params)

## 1. ¿Qué archivos hay?

Cada mes de prueba necesita tres meses **requeridos** (el mes de prueba, el mes anterior y el mismo mes del año anterior) y hasta cuatro **opcionales**. Los opcionales dan historia para los rezagos del mes de entrenamiento; si faltan, ese mes pierde sus primeros días (se ve en `dias_train`).

In [ ]:
archivos = MM.revisar_archivos(PRUEBAS)
display(archivos)
pendientes = archivos.loc[archivos["corre"], "prueba"].tolist()
print(f"Se pueden correr {len(pendientes)} de {len(PRUEBAS)} meses de prueba: {pendientes}")

### Comprobación de la limpieza

Este notebook lee los CSV mes por mes en lugar de usar `trips_clean.parquet`. Esta celda verifica que, para un mes que sí está en el parquet, salen **exactamente los mismos viajes** (mismo total y mismo conteo por estación). Se omite si no hay parquet.

In [ ]:
ruta_parquet = f"{M.PROCESSED_DIR}/trips_clean.parquet"
mes_chk = "2026-09"
if os.path.exists(ruta_parquet) and MM.disponible(mes_chk):
    ini, fin = pd.Timestamp(mes_chk + "-01"), pd.Timestamp(mes_chk + "-01") + pd.offsets.MonthEnd(0) + pd.Timedelta(days=1)
    p = pd.read_parquet(ruta_parquet, columns=["estacion_origen", "origen_dt", "duracion_min"])
    ok = p["duracion_min"].between(1, 45) & p["estacion_origen"].notna() & (p["origen_dt"] >= ini) & (p["origen_dt"] < fin)
    v = MM.validos_mes(mes_chk)
    v = v[(v["origen_dt"] >= ini) & (v["origen_dt"] < fin)]
    a = p[ok].groupby("estacion_origen").size().sort_index().astype(int)
    b = v.groupby("estacion_origen").size().sort_index().astype(int)
    print(f"{mes_chk}: parquet {int(ok.sum()):,} viajes | lectura por mes {len(v):,} | conteos por estación iguales: {a.equals(b)}")
    del p, v
else:
    print("Sin parquet o sin el archivo del mes: se omite la comprobación.")

## 2. Entrenamiento y evaluación

Para cada mes de prueba se entrena con (a) el mes anterior, (b) el mismo mes del año anterior y (c) ambos, y se predice el mes de prueba completo. La columna `segundos` guarda cuánto tardó cada modelo: sirve para decidir si hace falta Colab.

In [ ]:
previos = pd.read_csv(SALIDA) if (os.path.exists(SALIDA) and not REHACER) else pd.DataFrame()
hechos = set(previos["mes"]) if len(previos) else set()
filas = previos.to_dict("records")

t_total = time.time()
for prueba in pendientes:
    if prueba in hechos:
        print(f"{prueba}: ya estaba en {os.path.basename(SALIDA)}, se omite")
        continue
    print(f"\n=== {prueba}  (mismo mes, año anterior: {MM.plan_par(prueba)['mismo_mes_anio_anterior']}) ===")
    filas += MM.evaluar_par(prueba, params, estaciones, usar_lineal=USAR_LINEAL, usar_mlp=USAR_MLP)
    res = pd.DataFrame(filas)
    res.round(4).to_csv(SALIDA, index=False)          # se guarda después de cada mes
    if EN_COLAB and os.path.isdir(COPIA_A_DRIVE):
        res.round(4).to_csv(os.path.join(COPIA_A_DRIVE, os.path.basename(SALIDA)), index=False)
print(f"\nTiempo de esta corrida: {(time.time() - t_total) / 60:.1f} min")
res = pd.DataFrame(filas)

## 3. Resultados

In [ ]:
orden = ["Media 4 semanas",
         "Promedio histórico (mes anterior)", "Promedio histórico (mismo mes, año anterior)",
         "Regresión lineal (mes anterior)", "Regresión lineal (mismo mes, año anterior)",
         "MLP (mes anterior)", "MLP (mismo mes, año anterior)",
         "XGBoost (mes anterior)", "XGBoost (mismo mes, año anterior)", "XGBoost (ambos)"]
orden = [m for m in orden if m in res["modelo"].unique()]
for metrica in ["RMSE", "MAE", "Sesgo"]:
    t = res.pivot(index="modelo", columns="mes", values=metrica).loc[orden]
    t["Promedio"] = t.mean(axis=1)
    display(t.round(3).style.set_caption(metrica))

In [ ]:
# ¿Qué ayuda más? Diferencia de RMSE de XGBoost contra XGBoost (mes anterior); negativo = mejor que entrenar con el mes anterior
t = res.pivot(index="modelo", columns="mes", values="RMSE")
base = t.loc["XGBoost (mes anterior)"]
dif = (t.loc[[m for m in ["XGBoost (mismo mes, año anterior)", "XGBoost (ambos)"] if m in t.index]] - base).round(3)
dif["Promedio"] = dif.mean(axis=1).round(3)
display(dif.style.set_caption("RMSE de XGBoost menos RMSE de XGBoost (mes anterior). Negativo = mejor"))

dem = res.drop_duplicates("mes").set_index("mes")[["demanda_prueba"]].T.round(2)
display(dem.style.set_caption("Demanda media del mes de prueba (salidas por estación-hora)"))
dias = res[res["modelo"].isin(["XGBoost (mes anterior)", "XGBoost (mismo mes, año anterior)"])].pivot(
    index="modelo", columns="mes", values="dias_train")
display(dias.style.set_caption("Días de entrenamiento efectivos (menos de ~28 indica que faltó el mes previo para los rezagos)"))

In [ ]:
modelos_graf = [m for m in ["Promedio histórico (mes anterior)", "XGBoost (mes anterior)",
                            "XGBoost (mismo mes, año anterior)", "XGBoost (ambos)"] if m in res["modelo"].unique()]
t = res.pivot(index="mes", columns="modelo", values="RMSE")[modelos_graf]
fig, ax = plt.subplots(figsize=(11, 4.5))
t.plot.bar(ax=ax, width=0.8, rot=0)
ax.set_ylabel("RMSE (salidas por estación-hora)")
ax.set_xlabel("Mes de prueba")
ax.set_title("Un mes de entrenamiento: mes anterior contra mismo mes del año anterior")
ax.set_ylim(0, t.max().max() * 1.15)
ax.legend(fontsize=8, loc="lower left")
fig.tight_layout()
fig.savefig("../figures/20_mismo_mes_rmse.png", dpi=150)
plt.show()

In [ ]:
# Cuánto tarda: lo que decide si hace falta Colab
seg = res.groupby("mes")["segundos"].sum().round(0)
print("Segundos de entrenamiento por mes de prueba (sin contar la lectura de CSV):")
display(seg.to_frame("segundos").T)
print(f"Suma: {seg.sum() / 60:.1f} min con XGBoost en {dispositivo}")

## Lectura de resultados

*(Por completar después de correr los 13 meses.)*

Preguntas que hay que responder con la tabla de diferencias:

1. ¿Entrenar con el mismo mes del año anterior le gana al mes inmediato anterior? ¿En qué meses?
2. ¿Entrenar con los dos juntos mejora a ambos, o solo iguala al mejor?
3. ¿Cambia la conclusión entre los meses de 2026 (contra 2025) y los de 2025 (contra 2024)?
4. ¿Cuánto cambia la demanda media de un año al siguiente en el mismo mes? Si cambia mucho, el modelo "del año anterior" parte con el nivel equivocado, y los rezagos (`media_4sem`, `nivel_semana`) lo corrigen solo en parte.

**Limitaciones que ya se conocen:**
- Cada mes de entrenamiento tiene ~28 días, así que el modelo ve un solo ciclo mensual.
- El catálogo de estaciones (`stations_geo.csv`) es de 2026: las estaciones que ya no existen, o que se movieron, entran con latitud, longitud y capacidad vacías.
- Los hiperparámetros se eligieron con el periodo de 2026, así que favorecen un poco a los casos que entrenan con 2026.
- En los meses sin el archivo previo opcional, el mes de entrenamiento pierde sus primeros días por los rezagos (ver `dias_train`).